# 06 · Export validation: PyTorch → ONNX → TensorRT FP32

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1 (after Checkpoint 1 baseline is validated)

**Purpose.** Show that the exported graphs compute the same scores as PyTorch on the same saved frames, before any INT8 work.

**In plain words.** If the FP32 export already changed the scores, we could not tell export bugs from quantization effects. So we compare PyTorch, ONNX Runtime and TensorRT FP32 (with TF32 disabled) frame by frame.

**Expected inputs**
- Frozen checkpoint; processed dev evaluation frames.

**Expected outputs**
- `QPAD_Project/models/deeppixbis_fp32.onnx`
- `QPAD_Project/engines/fp32_tf32off.engine` (+ `.json`)
- `QPAD_Project/results/export/equivalence.json`, `failing_samples.csv`, per-backend frame scores

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
DRY_RUN_VIDEOS = 4      # dry run: frames of the first 4 dev videos
RUN_TENSORRT = True     # requires a CUDA GPU and TensorRT
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb06", paths.logs / "nb06.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

In [ ]:
# TensorRT is needed from here on. Its version is pinned in configs/int8_experiments.yaml: TensorRT 11
# removed the INT8 entropy calibrator that the pre-registered method uses. Colab preinstalls TensorRT 11,
# so the first time this cell swaps it for the pinned version and asks for ONE session restart.
TRT_PIN = str(cfgs["int8"]["tensorrt"]["version"])
def installed_trt():   # checked in a fresh process: an already-imported module would hide a reinstall
    r = subprocess.run([sys.executable, "-c", "import tensorrt; print(tensorrt.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None
if IN_COLAB and installed_trt() != TRT_PIN:
    import torch
    cuda_major = torch.version.cuda.split(".")[0]
    old = [p["name"] for p in json.loads(subprocess.run([sys.executable, "-m", "pip", "list", "--format=json"],
                                                        capture_output=True, text=True, check=True).stdout)
           if p["name"].lower().replace("_", "-").startswith("tensorrt")]
    if old:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *old], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"tensorrt-cu{cuda_major}=={TRT_PIN}"], check=True)
    if installed_trt() != TRT_PIN:
        raise RuntimeError(f"installing TensorRT {TRT_PIN} failed (found {installed_trt()})")
    if "tensorrt" in sys.modules:
        raise RuntimeError(f"TensorRT {TRT_PIN} is now installed. Runtime -> Restart session, then Run all again "
                           "(needed once per runtime; the old version is still loaded in this session).")
try:
    import tensorrt as trt
except ImportError:
    trt = None
print("TensorRT:", getattr(trt, "__version__", None))
if trt is not None and trt.__version__ != TRT_PIN:
    raise RuntimeError(f"TensorRT {trt.__version__} is loaded but {TRT_PIN} is pinned: restart the session.")
from qpad.tensorrt_utils import hardware_fingerprint
HW = hardware_fingerprint(); print(HW)

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
run_trt = RUN_TENSORRT and trt is not None and torch.cuda.is_available()
from qpad.experiments import export_validation, EquivalenceError
REPORT = export_validation(paths, cfgs, DRY_RUN, DEVICE, DRY_RUN_VIDEOS if DRY_RUN else None, run_trt,
                           tqdm_progress("scoring"))
print(json.dumps({k: REPORT[k] for k in ("n_frames", "n_videos", "pytorch_vs_onnx", "onnx_vs_trt_fp32", "passed")},
                 indent=1, default=str)[:4000])

In [ ]:
# STOP rule: never continue to INT8 if equivalence failed.
if not REPORT["passed"]:
    raise EquivalenceError("Export equivalence FAILED or TensorRT was not run. Inspect preprocessing, output tensor "
                           "(pixel_map), eval mode, ONNX operators and TensorRT flags. See failing_samples.csv.")
print("Equivalence PASSED: PyTorch vs ONNX <= 1e-5 and ONNX vs TensorRT FP32 (TF32 off) <= 1e-4.")

**Next notebook:** `07_int8_calibration_pilot.ipynb`